In [2]:
# Download dataset

import kagglehub
import os
from pathlib import Path


path = Path("../data") / "source"
os.makedirs(path, exist_ok=True)

if not os.listdir(path):
  kagglehub.dataset_download(
    "nadyinky/sephora-products-and-skincare-reviews",
    output_dir=path
  )

In [3]:
# Create db and bronze tables

import duckdb
from pathlib import Path

dir_data = Path("../data")
dir_data_source = dir_data / "source"

file_db = dir_data / "sephora.db"
file_products = dir_data_source / "product_info.csv"
files_reviews = dir_data_source / "reviews*.csv"

con = duckdb.connect(file_db)

con.execute(
    f"""
    CREATE OR REPLACE TABLE bronze_products AS
        SELECT *
        FROM read_csv(
            '{file_products}',
            header = true,
            delim = ','
        );

    CREATE OR REPLACE TABLE bronze_reviews AS
        SELECT *
        FROM read_csv(
            '{files_reviews}',
            header = true,
            delim = ','
        );
    """
)

In [4]:
con.sql("SELECT * FROM bronze_products LIMIT 10;")

┌────────────┬───────────────────────────────────────────┬──────────┬────────────┬─────────────┬────────┬─────────┬─────────────────┬────────────────────────────────────┬─────────────────────────────────────┬────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [5]:
con.sql("SELECT * FROM bronze_reviews LIMIT 10;")

┌──────────┬─────────────┬────────┬────────────────┬─────────────┬──────────────────────┬──────────────────────────┬──────────────────────────┬─────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────┬─────────────┬───────────┬─────────────┬────────────┬────────────┬────────────────────────────────────────────────────┬────────────┬───────────┐
│ column00 │  author_id  │ rating │ is_recommended │ helpfulness │ total_feedback_count │ total_neg_feedback_count │ total_pos_feedback_count │ submission_time │                                  

In [6]:
con.sql("DESCRIBE bronze_products;").show(max_rows=50)

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ product_id         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_name       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ brand_id           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ brand_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ loves_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ rating             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ reviews            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ size               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ variation_type     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │

In [7]:
con.sql("DESCRIBE bronze_reviews;").show(max_rows=50)

┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ column00                 │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ author_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rating                   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_recommended           │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ helpfulness              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ total_feedback_count     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ total_neg_feedback_count │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ total_pos_feedback_count │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ su

In [8]:
con.sql("""
    CREATE OR REPLACE TABLE silver_products AS
        SELECT
            * EXCLUDE (limited_edition, new , online_only, out_of_stock, sephora_exclusive , ingredients, highlights),
            CAST(limited_edition AS BOOLEAN) AS limited_edition,
            CAST(new AS BOOLEAN) AS new,
            CAST(online_only AS BOOLEAN) AS online_only,
            CAST(out_of_stock AS BOOLEAN) AS out_of_stock,
            CAST(sephora_exclusive AS BOOLEAN) AS sephora_exclusive,
            string_split(
                trim(
                    replace(highlights, $$'$$, ''),
                    '[]'
                ), ', '
            ) AS highlights,
            string_split(
                trim(
                    replace(ingredients, $$'$$, ''),
                    '[]'
                ), ', '
            ) AS ingredients
        FROM bronze_products;
    
    SELECT * FROM silver_products LIMIT 10;
""")

┌────────────┬───────────────────────────────────────────┬──────────┬────────────┬─────────────┬────────┬─────────┬─────────────────┬────────────────────────────────────┬─────────────────────────────────────┬────────────────┬───────────┬─────────────────┬────────────────┬──────────────────┬────────────────────┬───────────────────────────┬─────────────┬─────────────────┬─────────────────┬─────────────────┬─────────┬─────────────┬──────────────┬───────────────────┬──────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [27]:
con.sql("""
    CREATE OR REPLACE TABLE silver_reviews AS
        SELECT
            column00 AS review_id,
            * EXCLUDE (is_recommended),
            CAST(is_recommended AS BOOLEAN) AS is_recommended
        FROM bronze_reviews;
    
    SELECT * FROM silver_reviews LIMIT 10;
""")

┌───────────┬──────────┬─────────────┬────────┬─────────────┬──────────────────────┬──────────────────────────┬──────────────────────────┬─────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────┬─────────────┬───────────┬─────────────┬────────────┬────────────┬────────────────────────────────────────────────────┬────────────┬───────────┬────────────────┐
│ review_id │ column00 │  author_id  │ rating │ helpfulness │ total_feedback_count │ total_neg_feedback_count │ total_pos_feedback_count │ submission_time │                           

In [28]:
con.sql("""
    CREATE OR REPLACE TABLE gold_reviews_for_test_rag AS
        WITH product_review_stats AS (
            SELECT
                product_id
                ,count(*) AS num_reviews
                ,min(rating) AS min_rating
                ,max(rating) AS max_rating
                ,avg(rating) AS avg_rating
                -- ,review_title
                -- ,review_text
            FROM silver_reviews
            GROUP BY product_id
            HAVING (
                count(*) < 30
                AND count(*) > 25
                AND min(rating) < 2
                AND max(rating) > 4
            )
            ORDER BY abs(avg_rating - 2.5) ASC
            LIMIT 1
        )
        SELECT
            r.review_id
            ,r.product_id
            ,r.product_name
            ,r.rating
            ,r.is_recommended
            ,r.review_title
            ,r.review_text
        FROM silver_reviews r
            JOIN product_review_stats s
            ON r.product_id = s.product_id;
    SELECT * FROM gold_reviews_for_test_rag;
""").show(max_rows=50)

┌───────────┬────────────┬─────────────────────────────────────────┬────────┬────────────────┬─────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

def embed_text(text):
    return model.encode(text).tolist()

try:
    con.create_function("embed_text", embed_text, [str], list[float], )
except duckdb.NotImplementedException:
    pass

con.sql("""
    INSTALL vss;
    LOAD vss;
    SET hnsw_enable_experimental_persistence = TRUE;
            
    CREATE OR REPLACE TABLE reviews_test_rag AS
        SELECT
            *
            ,embed_text(review_title)::FLOAT[384] AS review_title_embedding
            ,embed_text(review_text)::FLOAT[384] AS review_text_embedding
        FROM gold_reviews_for_test_rag;
            
    SELECT * FROM reviews_test_rag;
""").show(max_rows=50)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9466.92it/s]


┌───────────┬────────────┬─────────────────────────────────────────┬────────┬────────────────┬─────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [61]:
def query_rag(con: duckdb.DuckDBPyConnection, embedding: list[float], table: str, array_length: str) -> None:
    con.sql(
        f"""
        SELECT
            * EXCLUDE (product_id, product_name, review_title, review_title_embedding, review_text_embedding)
        FROM {table}
        ORDER BY array_distance(review_text_embedding, ?::FLOAT[{array_length}]) ASC
        LIMIT 5;
    """, params=[embedding]).show()

In [62]:
query_rag(con, embed_text("I love this product, it works great and I highly recommend it."), "reviews_test_rag", "384")

┌───────────┬────────┬────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ review_id │ rating │ is_recommended │                                                                                                                                                                                                                                       

In [51]:
con.sql(
    """
    SELECT rating, review_text
    FROM reviews_test_rag
    ORDER BY rating DESC
    LIMIT 5;
""").show()

┌────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [52]:
query_rag(con, "I'm sucker for this product!")

┌───────────┬────────┬────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ review_id │ rating │ is_recommended │                                                                                                                                                                                                                                                    review_text                                                                                                                                                                      

In [55]:
from sentence_transformers import SentenceTransformer

model2 = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")

def embed_text2(text):
    return model2.encode(text).tolist()

try:
    con.create_function("embed_text2", embed_text2, [str], list[float], )
except duckdb.NotImplementedException:
    pass

con.sql("""
    INSTALL vss;
    LOAD vss;
    SET hnsw_enable_experimental_persistence = TRUE;
            
    CREATE OR REPLACE TABLE reviews_test_rag2 AS
        SELECT
            *
            ,embed_text2(review_title)::FLOAT[1024] AS review_title_embedding
            ,embed_text2(review_text)::FLOAT[1024] AS review_text_embedding
        FROM gold_reviews_for_test_rag;
            
    SELECT * FROM reviews_test_rag2;
""").show(max_rows=50)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7863.65it/s]


┌───────────┬────────────┬─────────────────────────────────────────┬────────┬────────────────┬─────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [63]:
query_rag(con, embed_text2("I love this product, it works great and I highly recommend it."), "reviews_test_rag2", "1024")

┌───────────┬────────┬────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [64]:
query_rag(con, embed_text2("Person is not happy with the product."), "reviews_test_rag2", "1024")

┌───────────┬────────┬────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ review_id │ rating │ is_recommended │                                                                                                                                                                                                                                       

In [65]:
query_rag(con, embed_text2("Person has mixed feeling about the product."), "reviews_test_rag2", "1024")

┌───────────┬────────┬────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ review_id │ rating │ is_recommended │                       

In [66]:
query_rag(con, embed_text2("Person thinks product is mediocre."), "reviews_test_rag2", "1024")

┌───────────┬────────┬────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ review_id │ rating │ is_recommended │                                                                                                                                                                                                                                       

In [67]:
query_rag(con, embed_text2("Person thinks product is really bad."), "reviews_test_rag2", "1024")

┌───────────┬────────┬────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ review_id │ rating │ is_recommended │                                                                                                                                                                                                                                       

In [68]:
con.close()